In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Đưa project root vào sys.path
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

# Quan trọng: đổi working directory
import os
os.chdir(ROOT)

from app.data import sync_data_from_s3
from helper.helper_functions import read_parquet

In [2]:
from app.config import settings

In [2]:
report = sync_data_from_s3()

In [3]:
print(f"Data ingestion report: {report}")

Data ingestion report: {'status': 'downloaded', 'bucket': 'baitap-buoi9-bucket', 'prefix': 'data/', 'files': ['attrition_labels.csv', 'employees.csv', 'engagement_surveys.csv', 'leave_logs.csv', 'performance_reviews.csv', 'work_activity.csv'], 'data_dir': 'data\\raw'}


In [3]:
df = read_parquet(settings.MERGED_DATA_PATH)

In [4]:
df.head()

,employee_id,snapshot_date,department,job_level,label,role,location,employment_type,hire_date,manager_id,...,peer_feedback_score,review_sentiment_score,week_start,tasks_completed,late_tasks,overtime_hours,meeting_hours,focus_hours,after_hours_messages,absent_days
0,E000001,2025-01-31,Customer Success,3,0,Senior CSM,Ho Chi Minh City,Full-time,2016-01-06,M00435,...,4.27,0.876,2025-01-27,61,7,9.5,11.9,28.4,17,0
1,E000002,2025-01-31,Operations,3,0,Senior Operations Specialist,Ho Chi Minh City,Full-time,2023-11-13,M00204,...,3.11,0.633,2025-01-27,47,1,8.5,10.6,26.5,8,0
2,E000003,2025-01-31,Marketing,2,0,Marketing Specialist,Ho Chi Minh City,Contract,2017-02-09,M00367,...,3.46,0.965,2025-01-27,27,2,8.7,7.9,23.0,9,0
3,E000004,2025-01-31,Customer Success,1,0,CS Associate,Da Nang,Full-time,2022-12-19,M00795,...,3.75,0.852,2025-01-27,52,4,10.9,1.3,27.9,12,0
4,E000005,2025-01-31,Sales,1,0,Sales Associate,Ha Noi,Full-time,2016-02-29,M00460,...,3.04,0.701,2025-01-27,51,5,10.4,4.7,31.0,4,0


In [14]:
df["department"].head()

0    Customer Success
1          Operations
2           Marketing
3    Customer Success
4               Sales
Name: department, dtype: str

In [4]:
import joblib
from app.config import settings

In [6]:
pipeline = joblib.load(settings.BEST_MODEL_PATH)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](32,)","['tasks_completed','late_tasks','overtime_hours',...,'role','location', 'employment_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,32
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough